## Refactoring is Calling

Here is another beautifully flawed Python script.
While the previous example suffered from "God Class" and architectural coupling, this one is an **ETL (Extract, Transform, Load)** script that falls into entirely different Python-specific traps. It’s functional, but it is a maintenance nightmare.

Now it is your turn in We Do session. You have *45min - 1h* to work on the problem

### 🛠️ Your New Refactoring Mission


- [ ] Don't start implementing immediately.
- [ ] First list the problems you see. There should be 6-9 categories.
- [ ] Iteratively fix them


In [4]:
import json
import csv

import pandas as pd

global_employee_database = []
total_processed = 0
errors_encountered = 0

def addEmployeeRecord(record, target_list=[]): 
    global total_processed
    global errors_encountered
    try:
        target_list.append(record)
        total_processed = total_processed + 1
    except Exception:
        pass

def parseDataFiles(file_list):
    if file_list != None:
        if len(file_list) > 0:
            for fileIndex in range(len(file_list)):
                fileName = file_list[fileIndex]
                if fileName.endswith(".csv"):
                    try:
                        f = open(fileName, 'r') 
                        lines = f.readlines()
                        for line in lines[1:]: 
                            parts = line.strip().split(',')
                            if len(parts) == 4:
                                empDict = {'id': parts[0], 'name': parts[1], 'dept': parts[2], 'salary': float(parts[3])}
                                if empDict['salary'] > 0:
                                    addEmployeeRecord(empDict)
                    except Exception:
                        global errors_encountered
                        errors_encountered += 1
                elif fileName.endswith(".json"):
                    try:
                        f2 = open(fileName, 'r') 
                        data = json.loads(f2.read())
                        for item in data:
                            if 'id' in item and 'name' in item and 'dept' in item and 'salary' in item:
                                if float(item['salary']) > 0:
                                    addEmployeeRecord(item)
                    except Exception:
                        errors_encountered += 1

def calculate_Bonus_and_Print_report():
    global global_employee_database
    
    engineering_employees = []
    sales_employees = []
    
    for i in range(len(global_employee_database)):
        if global_employee_database[i]['dept'] == 'Engineering':
            engineering_employees.append(global_employee_database[i])
        elif global_employee_database[i]['dept'] == 'Sales':
            sales_employees.append(global_employee_database[i])
            
    out_file = open("department_bonuses.txt", "w")

    
    out_file.write("--- ENGINEERING ---\n")
    for emp in engineering_employees:
        bonus = 0
        if emp['salary'] < 50000:
            bonus = emp['salary'] * 0.10
        elif emp['salary'] >= 50000 and emp['salary'] <= 100000:
            bonus = emp['salary'] * 0.05
        else:
            bonus = emp['salary'] * 0.02
            
        out_file.write(emp['name'] + " gets a bonus of $" + str(bonus) + "\n")
        
    out_file.write("--- SALES ---\n")
    for emp in sales_employees:
        bonus = 0
        if emp['salary'] < 50000:
            bonus = emp['salary'] * 0.15 
        elif emp['salary'] >= 50000 and emp['salary'] <= 100000:
            bonus = emp['salary'] * 0.07
        else:
            bonus = emp['salary'] * 0.03
        out_file.write(emp['name'] + " gets a bonus of $" + str(bonus) + "\n")


if __name__ == '__main__':
    print("Starting batch job...")
    
    with open('data1.csv', 'w') as f:
        f.write("id,name,dept,salary\n1,Alice,Engineering,60000\n2,Bob,Sales,45000\n")
    with open('data2.json', 'w') as f:
        f.write('[{"id": 3, "name": "Charlie", "dept": "Engineering", "salary": 120000}]')

    files_to_process = ['data1.csv', 'data2.json', 'missing_file.csv']
    
    parseDataFiles(files_to_process)
    calculate_Bonus_and_Print_report()
    
    print("Job Done. Processed: " + str(total_processed) + ", Errors: " + str(errors_encountered))


Starting batch job...
Job Done. Processed: 3, Errors: 1


## Round 1

In [21]:
import json
import csv

import pandas as pd
from dataclasses import dataclass
from enum import StrEnum

"""
+ Naming issues addEmployeeRecord -> add_employee_record
+ employee as dict to Employee as dataclass
+ Nested ifs to short circuit
+ File handler leakage
+ No need for loads --> load will take care file handler.
+ Temporarily fix a bug that will pass global database.
+ Convert literal department string to enum based one.
+ List comphrension in report
"""

global_employee_database = []
total_processed = 0
errors_encountered = 0


class DepartmentType(StrEnum):
    SALES = "Sales"
    ENGINEERING = "Engineering"


@dataclass(frozen=True)
class Employee:
    id: int = None
    name: str = None
    dept: DepartmentType = None
    salary: float = None

    @classmethod
    def from_dict(cls, d: dict) -> "Employee":
        return Employee(id=int(d.get("id")), name=d.get("name"), dept=d.get("dept"), salary=float(d.get("salary")))


def add_employee_record(record: Employee, target_list: list[Employee] = []):
    global total_processed
    global errors_encountered
    try:
        target_list.append(record)
        total_processed = total_processed + 1
    except Exception:
        pass


def parse_data_files(file_list):
    global global_employee_database
    if not file_list:
        return

    if len(file_list) == 0:
        return

    for file_name in file_list:
        if file_name.endswith(".csv"):
            try:
                with open(file_name, 'r') as fp:
                    _ = next(fp)  # Skip header
                    for line in fp: 
                        parts = line.strip().split(',')
                        if len(parts) == 4:
                            empDict = {'id': parts[0], 'name': parts[1], 'dept': parts[2], 'salary': float(parts[3])}
                            e = Employee.from_dict(empDict)
                            if e.salary > 0:
                                add_employee_record(e, global_employee_database)
            except Exception:
                global errors_encountered
                errors_encountered += 1

        elif file_name.endswith(".json"):
            try:
                with open(file_name, 'r') as fp:
                    data = json.load(fp)

                for item in data:
                    e = Employee.from_dict(item)
                    if e.id is not None and e.name is not None and e.dept is not None and e.salary is not None:
                        if e.salary > 0:
                            add_employee_record(e, global_employee_database)
            except Exception:
                errors_encountered += 1


def calculate_bonus_and_print_report():
    global global_employee_database

    engineering_employees = [emp for emp in global_employee_database if emp.dept == DepartmentType.ENGINEERING]
    sales_employees = [emp for emp in global_employee_database if emp.dept == DepartmentType.SALES]

    with open("department_bonuses.txt", "w") as out_file:
        out_file.write("--- ENGINEERING ---\n")
        for emp in engineering_employees:
            bonus = 0
            if emp.salary < 50000:
                bonus = emp.salary * 0.10
            elif emp.salary >= 50000 and emp.salary <= 100000:
                bonus = emp.salary * 0.05
            else:
                bonus = emp.salary * 0.02

            out_file.write(emp.name + " gets a bonus of $" + str(bonus) + "\n")

        out_file.write("--- SALES ---\n")
        for emp in sales_employees:
            bonus = 0
            if emp.salary < 50000:
                bonus = emp.salary * 0.15 
            elif emp.salary >= 50000 and emp.salary <= 100000:
                bonus = emp.salary * 0.07
            else:
                bonus = emp.salary * 0.03
            out_file.write(emp.name + " gets a bonus of $" + str(bonus) + "\n")


if __name__ == '__main__':
    print("Starting batch job...")

    with open('data1.csv', 'w') as f:
        f.write("id,name,dept,salary\n1,Alice,Engineering,60000\n2,Bob,Sales,45000\n")
    with open('data2.json', 'w') as f:
        f.write('[{"id": 3, "name": "Charlie", "dept": "Engineering", "salary": 120000}]')

    files_to_process = ['data1.csv', 'data2.json', 'missing_file.csv']

    parse_data_files(files_to_process)
    calculate_bonus_and_print_report()

    print("Job Done. Processed: " + str(total_processed) + ", Errors: " + str(errors_encountered))

Starting batch job...
Job Done. Processed: 3, Errors: 1


In [23]:
!cat department_bonuses.txt

--- ENGINEERING ---
Alice gets a bonus of $3000.0
Charlie gets a bonus of $2400.0
--- SALES ---
Bob gets a bonus of $6750.0


## Round 2

In [38]:
import json
import csv

import pandas as pd
from dataclasses import dataclass
from enum import StrEnum

"""
- Remove global state variable for database
- Remove processing state variable
- SRP for print_bonus_report
"""

class DepartmentType(StrEnum):
    SALES = "Sales"
    ENGINEERING = "Engineering"


@dataclass(frozen=True)
class Employee:
    id: int = None
    name: str = None
    dept: DepartmentType = None
    salary: float = None

    @classmethod
    def from_dict(cls, d: dict) -> "Employee":
        return Employee(id=int(d.get("id")), name=d.get("name"), dept=d.get("dept"), salary=float(d.get("salary")))

    def bonus(self) -> float:
        bonus = 0
        if self.dept == DepartmentType.ENGINEERING:
            if self.salary < 50000:
                bonus = self.salary * 0.10
            elif self.salary >= 50000 and self.salary <= 100000:
                bonus = self.salary * 0.05
            else:
                bonus = self.salary * 0.02
        else:
            if self.salary < 50000:
                bonus = self.salary * 0.15 
            elif self.salary >= 50000 and self.salary <= 100000:
                bonus = self.salary * 0.07
            else:
                bonus = self.salary * 0.03

        return bonus
        


class ProcessingState:
    def __init__(self):
        self.success_records = 0
        self.errors = 0

    def ok(self):
        self.success_records += 1

    def nok(self):
        self.errors += 1

    def show_report(self):
        print(f"Job Done. Processed: {self.success_records}, Errors: {self.errors}")


class EmployeeDB:
    def __init__(self, processing_state: ProcessingState):
        self.employees = []
        self.processing_state = processing_state

    def add(self, e: Employee):
        try:
            self.employees.append(e)
            self.processing_state.ok()
        except Exception:
            pass
        finally:
            return self

    def filter(self, dept_filter: DepartmentType) -> list[Employee]:
        return [emp for emp in self.employees if emp.dept == dept_filter]


def parse_data_files(file_list) -> EmployeeDB:
    pstate = ProcessingState()
    db = EmployeeDB(pstate)

    if not file_list:
        return db

    if len(file_list) == 0:
        return db
        
    for file_name in file_list:
        if file_name.endswith(".csv"):
            try:
                with open(file_name, 'r') as fp:
                    _ = next(fp)  # Skip header
                    for line in fp: 
                        parts = line.strip().split(',')
                        if len(parts) == 4:
                            empDict = {'id': parts[0], 'name': parts[1], 'dept': parts[2], 'salary': float(parts[3])}
                            e = Employee.from_dict(empDict)
                            if e.salary > 0:
                                db.add(e)
            except Exception:
                pstate.nok()

        elif file_name.endswith(".json"):
            try:
                with open(file_name, 'r') as fp:
                    data = json.load(fp)

                for item in data:
                    e = Employee.from_dict(item)
                    if e.id is not None and e.name is not None and e.dept is not None and e.salary is not None:
                        if e.salary > 0:
                            db.add(e)
            except Exception:
                pstate.nok()
    return db


def print_bonus_report(db:EmployeeDB):
    with open("department_bonuses.txt", "w") as out_file:
        out_file.write("--- ENGINEERING ---\n")
        for emp in db.filter(DepartmentType.ENGINEERING):
            out_file.write(f"{emp.name} gets a bonus of ${emp.bonus()}\n")

        out_file.write("--- SALES ---\n")
        for emp in db.filter(DepartmentType.SALES):
            out_file.write(f"{emp.name} gets a bonus of ${emp.bonus()}\n")

if __name__ == '__main__':
    print("Starting batch job...")

    with open('data1.csv', 'w') as f:
        f.write("id,name,dept,salary\n1,Alice,Engineering,60000\n2,Bob,Sales,45000\n")
    with open('data2.json', 'w') as f:
        f.write('[{"id": 3, "name": "Charlie", "dept": "Engineering", "salary": 120000}]')

    files_to_process = ['data1.csv', 'data2.json', 'missing_file.csv']

    db = parse_data_files(files_to_process)
    print_bonus_report(db)

    db.processing_state.show_report()



Starting batch job...
Job Done. Processed: 3, Errors: 1


In [40]:
!cat department_bonuses.txt

--- ENGINEERING ---
Alice gets a bonus of $3000.0
Charlie gets a bonus of $2400.0
--- SALES ---
Bob gets a bonus of $6750.0


## Round 3

In [56]:
import json
import csv

import pandas as pd
from dataclasses import dataclass
from enum import StrEnum
from typing import Protocol
import traceback


class DepartmentType(StrEnum):
    SALES = "Sales"
    ENGINEERING = "Engineering"


@dataclass(frozen=True)
class Employee:
    id: int = None
    name: str = None
    dept: DepartmentType = None
    salary: float = None

    @classmethod
    def from_dict(cls, d: dict) -> "Employee":
        return Employee(id=int(d.get("id")), name=d.get("name"), dept=d.get("dept"), salary=float(d.get("salary")))

    def bonus(self) -> float:
        bonus = 0
        if self.dept == DepartmentType.ENGINEERING:
            if self.salary < 50000:
                bonus = self.salary * 0.10
            elif self.salary >= 50000 and self.salary <= 100000:
                bonus = self.salary * 0.05
            else:
                bonus = self.salary * 0.02
        else:
            if self.salary < 50000:
                bonus = self.salary * 0.15 
            elif self.salary >= 50000 and self.salary <= 100000:
                bonus = self.salary * 0.07
            else:
                bonus = self.salary * 0.03

        return bonus

    def is_valid(self) -> bool:
        return not self.id and not self.name and not self.dept and not self.salary and  self.salary > 0
        


class ProcessingState:
    def __init__(self):
        self.success_records = 0
        self.errors = 0

    def ok(self):
        self.success_records += 1

    def nok(self, e: Exception):
        self.errors += 1
        print(f"Error in processing {e}")
        traceback.print_exc()

    def show_report(self):
        print(f"Job Done. Processed: {self.success_records}, Errors: {self.errors}")


class EmployeeDB:
    def __init__(self, processing_state: ProcessingState):
        self.employees = []
        self.processing_state = processing_state

    def add(self, e: Employee):
        try:
            self.employees.append(e)
            self.processing_state.ok()
        except Exception:
            pass
        finally:
            return self

    def filter(self, dept_filter: DepartmentType) -> list[Employee]:
        return [emp for emp in self.employees if emp.dept == dept_filter]


class FileHandler(Protocol):
    def employees(self, pstate: ProcessingState=None)-> list[Employee]:
        ...

class CSVFile:
    def __init__(self,file_name, seperator = ","):
        self.file_name = file_name
        self.seperator = seperator 

    def employees(self, pstate: ProcessingState=None)-> list[Employee]:
        result = []
        try:
            with open(self.file_name, 'r') as fp:
                reader = csv.DictReader(self.file_name, delimiter=self.seperator)

                for rec in reader:
                    e = Employee.from_dict(rec)

                    if e.is_valid:
                        result.append(e)
        except Exception as err:
            if pstate:
                pstate.nok(err)
        finally:
            return result

class JsonFile:
    def __init__(self,file_name):
        self.file_name = file_name
        
    def employees(self, pstate: ProcessingState=None)-> list[Employee]: 
        result = []
        try:
            with open(self.file_name, 'r') as fp:
                data = json.load(fp)

            for item in data:
                e = Employee.from_dict(item)
                
                if e.is_valid:
                    result.append(e)
                        
        except Exception as err:
            if pstate:
                pstate.nok(err)
        finally:
            return result
        

def get_file_handler(file_name)->FileHandler:
    if file_name.endswith(".csv"):
        return CSVFile(file_name)
    elif file_name.endswith(".json"):
        return JsonFile(file_name) 
    else:
        raise ValueError(f"No handler found for file {file_name}")

def parse_data_files(file_list) -> EmployeeDB:
    pstate = ProcessingState()
    db = EmployeeDB(pstate)

    if not file_list:
        return db

    if len(file_list) == 0:
        return db

    for file_name in file_list:

        try:

            handler = get_file_handler(file_name)
    
            for emp in handler.employees(pstate):
                db.add(emp)
            
        except Exception as err:
            pstate.nok(err)
        
    
    return db


def print_bonus_report(db:EmployeeDB):
    with open("department_bonuses.txt", "w") as out_file:
        out_file.write("--- ENGINEERING ---\n")
        for emp in db.filter(DepartmentType.ENGINEERING):
            out_file.write(f"{emp.name} gets a bonus of ${emp.bonus()}\n")

        out_file.write("--- SALES ---\n")
        for emp in db.filter(DepartmentType.SALES):
            out_file.write(f"{emp.name} gets a bonus of ${emp.bonus()}\n")
        

if __name__ == '__main__':
    print("Starting batch job...")

    with open('data1.csv', 'w') as f:
        f.write("id,name,dept,salary\n1,Alice,Engineering,60000\n2,Bob,Sales,45000\n")
    with open('data2.json', 'w') as f:
        f.write('[{"id": 3, "name": "Charlie", "dept": "Engineering", "salary": 120000}]')

    files_to_process = ['data1.csv', 'data2.json', 'missing_file.csv']

    db = parse_data_files(files_to_process)
    print_bonus_report(db)

    db.processing_state.show_report()



Starting batch job...
{'d': 'a'}
Error in processing int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
Error in processing [Errno 2] No such file or directory: 'missing_file.csv'
Job Done. Processed: 1, Errors: 2


Traceback (most recent call last):
  File "/tmp/ipykernel_902/3573928385.py", line 103, in employees
    e = Employee.from_dict(rec)
        ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_902/3573928385.py", line 25, in from_dict
    return Employee(id=int(d.get("id")), name=d.get("name"), dept=d.get("dept"), salary=float(d.get("salary")))
                       ^^^^^^^^^^^^^^^^
TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
Traceback (most recent call last):
  File "/tmp/ipykernel_902/3573928385.py", line 98, in employees
    with open(self.file_name, 'r') as fp:
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 324, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'missing_file.csv'


In [50]:
!cat data1.csv

id,name,dept,salary
1,Alice,Engineering,60000
2,Bob,Sales,45000
